# Principal Component Analysis

**Companion lesson:** https://ml-viz.vercel.app/courses/pca-dimensionality/01-pca

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## PCA via Eigendecomposition

1. Center the data
2. Compute covariance matrix
3. Eigendecompose
4. Project onto top-k eigenvectors

In [ ]:
np.random.seed(42)
n = 200
t = np.linspace(0, 2 * np.pi, n)
X = np.column_stack([3 * np.cos(t), 1.5 * np.sin(t)]) + np.random.randn(n, 2) * 0.3

X_centered = X - X.mean(axis=0)
cov = np.cov(X_centered.T)
eigenvalues, eigenvectors = np.linalg.eigh(cov)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

X_pca = X_centered @ eigenvectors[:, :2]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Original data with principal components
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], c='#818cf8', s=10, alpha=0.5)
mean = X.mean(axis=0)
for i in range(2):
    v = eigenvectors[:, i] * np.sqrt(eigenvalues[i]) * 2
    ax.arrow(mean[0], mean[1], v[0], v[1], head_width=0.2, head_length=0.1,
             fc='#f43f5e' if i == 0 else '#14b8a6', ec='white', linewidth=1.5)
ax.set_title('Original + PCs', color='white', fontsize=11)
ax.set_aspect('equal')

# Projected onto PC1
ax = axes[1]
ax.scatter(X_pca[:, 0], np.zeros_like(X_pca[:, 0]), c='#818cf8', s=10, alpha=0.5)
ax.set_title('Projected onto PC1', color='white', fontsize=11)
ax.set_xlabel('$z_1$')

# Explained variance
ax = axes[2]
var_ratio = eigenvalues / eigenvalues.sum()
ax.bar(range(1, len(var_ratio) + 1), var_ratio, color=['#f43f5e', '#14b8a6', '#eab308'][:len(var_ratio)])
ax.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), 'o-', color='white', linewidth=1.5)
ax.set_xlabel('Component')
ax.set_ylabel('Variance Explained')
ax.set_title('Scree Plot', color='white', fontsize=11)
ax.axhline(0.95, color='#94a3b8', linestyle='--', alpha=0.5, label='95% threshold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'PC1 explains {var_ratio[0]*100:.1f}% of variance')
print(f'PC1+PC2 explain {sum(var_ratio[:2])*100:.1f}% of variance')

## Why PC1 is the top eigenvector — by hand

The variance along a unit direction $v$ is $\mathrm{Var}(\tilde X v) = v^\top C v$. Maximizing it under $\lVert v\rVert=1$ with a Lagrange multiplier gives $Cv=\lambda v$ — so $v$ is an eigenvector of the covariance and the captured variance equals $\lambda$. Pick the **largest** $\lambda$ for PC1.

Below we take the lesson's 4-point example, get the eigenvalues straight from the characteristic polynomial $\lambda^2-(\mathrm{tr}\,C)\lambda+\det C=0$, and confirm the projected variance equals the eigenvalue and that `sklearn` agrees.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

Xe = np.array([[2, 1], [1, 2], [-1, -2], [-2, -1]], dtype=float)  # already centered
n = len(Xe)
C = (Xe.T @ Xe) / (n - 1)                 # = np.cov(Xe.T)
print('covariance C =\n', np.round(C, 3))

# Eigenvalues by hand: lambda^2 - tr*lambda + det = 0  (2x2 quadratic formula)
tr, det = np.trace(C), np.linalg.det(C)
disc = np.sqrt(tr**2 - 4 * det)
lam = sorted([(tr + disc) / 2, (tr - disc) / 2], reverse=True)
print(f'tr={tr:.3f}, det={det:.3f}  ->  lambda1={lam[0]:.3f}, lambda2={lam[1]:.3f}')
print('eigvals (np.linalg.eigh):', np.round(np.linalg.eigh(C)[0][::-1], 3))

# Eigenvector for lambda1 from (C - lambda*I) v = 0, then verify projected variance == lambda
w, V = np.linalg.eigh(C)
v1 = V[:, -1]                              # eigenvector of largest eigenvalue
print('\nPC1 direction (normalized):', np.round(v1 / np.abs(v1).max(), 3), ' (proportional to (1,1))')
proj_var = np.var(Xe @ v1, ddof=1)
print(f'variance of projection onto PC1 = {proj_var:.3f}  ==  lambda1 = {lam[0]:.3f}')
print(f'variance ratio kept by PC1 = {lam[0] / (lam[0] + lam[1]):.3f}  (90%)')

# sklearn cross-check
pca = PCA(n_components=2).fit(Xe)
print('\nsklearn explained_variance_:', np.round(pca.explained_variance_, 3))
print('sklearn variance ratio     :', np.round(pca.explained_variance_ratio_, 3))


## Explained variance and the scree plot

Each principal component captures a share of the total variance. The cumulative curve tells you how many components to keep.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits

X = load_digits().data            # 64 features
pca = PCA().fit(X)
cum = np.cumsum(pca.explained_variance_ratio_)
k95 = np.argmax(cum >= 0.95) + 1
print(f'{k95} components explain 95% of variance (of {X.shape[1]})')

plt.plot(range(1, len(cum)+1), cum, color='#818cf8')
plt.axhline(0.95, ls='--', color='#f43f5e'); plt.axvline(k95, ls='--', color='#14b8a6')
plt.xlabel('components'); plt.ylabel('cumulative explained variance')
plt.title('Scree / cumulative variance'); plt.show()

## Key takeaways

- PCA finds orthogonal directions (**eigenvectors of the covariance**) of maximum variance.
- Projecting onto the top $k$ components reduces dimensions while keeping most variance.
- Choose $k$ from the **cumulative explained variance** (e.g. 95%) or the scree elbow.
- **Standardize first**; PCA is linear and unsupervised, computed efficiently via **SVD**.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The covariance matrix

PCA starts from the covariance of **centered** data:

$$C = \frac{1}{n} X_c^\top X_c, \qquad X_c = X - \bar{X}$$

Implement it. The checks verify it against `np.cov`, its symmetry, and that the diagonal holds the per-feature variances.

In [ ]:
def covariance(X):
    """Covariance matrix (population, 1/n) of the rows of X."""
    X = np.asarray(X, dtype=float)

    # TODO(you): subtract the column means
    Xc = ...

    # TODO(you): Xc^T Xc / n
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(0)
Xp = rng.multivariate_normal([1, -2], [[3, 1], [1, 2]], size=500)

C = covariance(Xp)
assert C.shape == (2, 2) and np.allclose(C, C.T), "covariance is square and symmetric"
assert np.allclose(C, np.cov(Xp.T, ddof=0)), "must match np.cov with ddof=0"
assert np.allclose(np.diag(C), Xp.var(axis=0)), "the diagonal holds the per-feature variances"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def covariance(X):
    X = np.asarray(X, dtype=float)
    Xc = X - X.mean(axis=0)
    return Xc.T @ Xc / len(X)
```

</details>

### Exercise 2 — Explained variance and choosing m

Each eigenvalue is the variance captured along its component, so the **explained variance ratio** is the sorted eigenvalues divided by their sum — and "how many components for 90%?" is just the first index where the cumulative sum crosses the threshold. Implement both halves of the scree-plot workflow.

In [ ]:
def explained_variance_ratio(eigvals):
    """Eigenvalues sorted descending, normalized to sum to 1."""
    lams = np.sort(np.asarray(eigvals, dtype=float))[::-1]

    # TODO(you): normalize
    return ...


def components_for(eigvals, threshold):
    """Smallest m whose top-m components explain >= threshold of the variance."""
    evr = explained_variance_ratio(eigvals)

    # TODO(you): first index where the cumulative sum reaches threshold, plus 1
    # (hint: np.cumsum + np.searchsorted)
    return ...

In [ ]:
# Checks — run me
assert np.allclose(explained_variance_ratio([1.0, 3.0]), [0.75, 0.25]), "sorted descending, normalized"
assert abs(sum(explained_variance_ratio([5, 2, 1, 0.5])) - 1) < 1e-12, "ratios sum to 1"
assert components_for([5.0, 3.0, 1.0, 1.0], 0.8) == 2, "5+3 of 10 covers 80% at m=2"
assert components_for([5.0, 3.0, 1.0, 1.0], 0.95) == 4, "the tail matters for 95%"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def explained_variance_ratio(eigvals):
    lams = np.sort(np.asarray(eigvals, dtype=float))[::-1]
    return lams / lams.sum()


def components_for(eigvals, threshold):
    evr = explained_variance_ratio(eigvals)
    return int(np.searchsorted(np.cumsum(evr), threshold) + 1)
```

</details>